In [1]:
import json
from pathlib import Path
import jsonlines
from src.data.utils import generation_task_from_json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import plotly.express as px

use_submit = True

if use_submit:
    files = Path('data/submit_with_metrics').glob('*.jsonl')
    submit_tasks = [generation_task_from_json(x) for x in jsonlines.open('data/reference_taskB.jsonl')]
else:
    files = Path('data/old_set/metrics').glob('*_test96.jsonl')
    split_ids = json.loads(Path('splits/test_96.json').read_text())
    submit_tasks = [
        generation_task_from_json(x)
        for x in jsonlines.open('/home/oleg/rag_workspace/mt-rag-benchmark/human/generation_tasks/reference.jsonl')
        if x['task_id'] in split_ids
    ]

submit_data = {
    path.stem: {x['task_id']: generation_task_from_json(x) for x in jsonlines.open(path)}
    for path in files
}
model_names = list(submit_data)

submit_ids = np.array([x.task_id for x in submit_tasks])
is_empty_context = np.array([len(x.contexts) == 0 for x in submit_tasks])

rl_f_dicts = {
    name: {x.task_id: x.metrics.RL_F for x in tasks_dict.values()}
    for name, tasks_dict in submit_data.items()
}

rl_f_lists = {
    name: [dict_.get(task_id, np.nan) for task_id in submit_ids]
    for name, dict_ in rl_f_dicts.items()
}

len_dicts = {
    name: {x.task_id: len(x.predictions[0].text) for x in tasks_dict.values()}
    for name, tasks_dict in submit_data.items()
}

len_lists = {
    name: [dict_.get(task_id, np.nan) for task_id in submit_ids]
    for name, dict_ in len_dicts.items()
}

# axis 0: model name (labels: model_names)
# axis 1: task id (labels: submit_ids)
rl_f_matrix = np.array([rl_f_lists[name] for name in model_names], dtype=float)
# print(sum(np.isnan(rl_f_matrix[3])))
rl_f_matrix = rl_f_matrix[:, ~is_empty_context]

submit_ids = submit_ids[~is_empty_context]

len_matrix = np.array([len_lists[name] for name in model_names], dtype=float)
len_matrix = len_matrix[:, ~is_empty_context]

avg_rl_f = np.nanmean(rl_f_matrix, axis=1)

top_model_idx_per_sample = (
    rl_f_matrix + avg_rl_f[:, None] * 0.0001
    # rl_f_matrix + len_matrix * 0.0000001
).argmax(axis=0)

{
    model_names[model_idx]: count
    for model_idx, count in Counter(top_model_idx_per_sample).items()
}

{'submit_gemini-3-pro-preview-high_new_prompt': 321,
 'submit_glm_46_gemini_prompt': 34,
 'submit_llama3_3_70b_gemini_prompt': 4,
 'submit_haiku45_taskB': 14,
 'submit_qwen2_5_32b_taskB': 2,
 'submit_qwen235b_gemini_prompt_no_empty': 2}

In [5]:
ensemble_preds = {}
top_models = {}

for task_idx, task_id in enumerate(submit_ids):
    top_model = model_names[int(top_model_idx_per_sample[task_idx])]
    top_model_predict = submit_data[top_model][task_id].predictions[0].text
    ensemble_preds[task_id] = top_model_predict
    top_models[task_id] = top_model

In [15]:
list(submit_ids).index('378b9c8fa2b9e8d571160d52f8392ede<::>2')

47

In [16]:
rl_f_matrix[:, 47]

array([1.        , 0.77777778, 0.76470588, 0.88      , 1.        ,
       1.        , 1.        ])

In [17]:
model_names

['submit_haiku45_taskB',
 'submit_qwen235b_gemini_prompt_no_empty',
 'submit_llama3_3_70b_gemini_prompt',
 'submit_gemini-3-pro-preview-high_new_prompt',
 'submit_glm_46_gemini_prompt',
 'submit_meno_v17_ckp420_taskB',
 'submit_qwen2_5_32b_taskB']

In [18]:
top_model_idx_per_sample[47]

4

In [13]:
submit_to_check = list(jsonlines.open('submit_ensemble.jsonl'))

for row in submit_to_check:
    task_id = row['task_id']
    pred = row['predictions'][0]['text']
    if task_id in ensemble_preds:
        if ensemble_preds[task_id] != pred:
            print(task_id, top_models[task_id])
            raise ValueError()

378b9c8fa2b9e8d571160d52f8392ede<::>2 submit_glm_46_gemini_prompt


ValueError: 

In [ ]:
    import jsonlines
    from pathlib import Path
    import json

    data = list(jsonlines.open('submit_ensemble.jsonl'))
    for x in data:
        if x['contexts'] == []:
            x['predictions'] = [{'text': "I don't have an answer."}]
    Path('submit_ensemble_NEW.jsonl').write_text('\n'.join(
        json.dumps(x, ensure_ascii=True) for x in data
    ))

3836831

In [8]:
data = list(jsonlines.open('submit_ensemble_NEW.jsonl'))
for x in data:
    if x['contexts'] != []:
        assert x['predictions'] != [{'text': "I don't have an answer."}]

In [ ]:

# plt.figure(figsize=(10, 3))
# plt.imshow(rl_f_matrix, aspect='auto', interpolation='none')
# plt.gca().set_yticks(np.arange(len(model_names)))
# plt.gca().set_yticklabels([f'{name} [{avg:.3f}]' for name, avg in zip(model_names, avg_rl_f)]);


# is_better = np.full((len(model_names), len(model_names)), np.nan)
# for model1_idx, model1 in enumerate(model_names):
#     for model2_idx, model2 in enumerate(model_names):
#         if model1_idx != model2_idx:
#             val1 = rl_f_matrix[model1_idx]
#             val2 = rl_f_matrix[model2_idx]
#             is_better[model1_idx, model2_idx] = np.sum(val1 > val2)

# px.imshow(
#     is_better, text_auto=True,
#     color_continuous_scale='Viridis',
#     x=model_names, y=model_names
# )

array([5])